In [ ]:
# Daily Challenge: Pinecone Serverless Reranking in Action


# Why are we doing this?

# Reranking models boost search relevance by assigning similarity scores between a query and documents, then
# reordering results so the most pertinent information appears first. In contexts like healthcare, this helps clinicians
# quickly access the most critical clinical notes.


# Task Overview & Detailed Explanations

# Below is a skeleton pipeline. Each numbered item is an action you must complete. After every instruction, you’ll find a
# clear explanation of what to do and why it’s important. Whenever you see … , replace it with the appropriate code or
# value, using the hint for guidance.

# ⚠️ Important: Make sure you have a Pinecone account and API key ready. Sign up at pinecone if you haven’t already.


# Part 1: Load Documents & Execute Reranking Model


# 1. Install Pinecone libraries

# !pip install -U pinecone==6.0.1 pinecone-notebooks


#     What to do: Run this command in your notebook cell (note the ! for notebook execution).
#     Why: You’ll need the client package to interact with Pinecone’s API and the notebook helper to simplify authentication in environments like Colab.
#     💡 Hint: If you get version conflicts, restart your runtime after installation.


# 2. Authenticate with Pinecone

# import os
# if not os.environ.get("PINECONE_API_KEY"):
#     from pinecone_notebooks.colab import Authenticate
#     Authenticate()


#     What to do: Run this code block exactly as shown. It will prompt for your API key if not already set.
#     Why: Securely providing your API key lets the client connect to your Pinecone project without hard-coding secrets in your script.
#     💡 Hint: Get your API key from your Pinecone dashboard under “API Keys”. Keep it secret!


# 3. Instantiate the Pinecone client

# from pinecone import Pinecone
# api_key = os.environ.get("PINECONE_API_KEY")
# pc = Pinecone(api_key=api_key)


#     What to do: Copy this code exactly - no need to fill in … here! The modern Pinecone client auto-detects your environment.
#     Why: The client ( pc ) is your entry point for all Pinecone operations—creating indexes, querying, and reranking.
#     💡 Hint: If you get authentication errors, make sure your API key is correct and active.


# 4. Define your query & documents

# query = "Tell me about Apple's products"
# documents = [
#     "...", # Add a document about apple fruit
#     "...", # Add a document about Apple company products
#     "...", # Add another fruit-related document
#     "...", # Add another company-related document
#     "..." # Add one more document (your choice)
# ]


#     What to do: Replace each ... with actual text documents that mix references to Apple (company) and apple (fruit).
#     Why: You need a small set of documents to test the reranker’s ability to distinguish between different contexts of the same word.
#     💡 Hint: Make some about “Apple is a fruit” and others about “Apple makes iPhones” - this tests contextual
#     understanding.


# 5. Call the reranker

# from pinecone import RerankModel
# reranked = pc.inference.rerank(
#     model="bge-reranker-v2-m3",
#     query=query,
#     documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
#     top_n=... # e.g., 3
# )


#     What to do: Fill in top_n with how many top results you want returned (e.g., 3).
#     Why: top_n limits the number of reranked results, so you only retrieve the most relevant documents.
#     💡 Hint: Try different top_n values to see how the ranking changes!


# 6. Inspect reranked results

# def show_reranked_results(query, matches):
#     print(f"Query: {query}")
#     for i, m in enumerate(matches):
#         ... # Print the position (i+1), m.score, and m.document.text

# show_reranked_results(query, reranked....) # Fill in the correct attribute


#     What to do: Replace ... with code that prints out the rank (i+1), the similarity score m.score, and the document text m.document.text . Also fill in the correct attribute for reranked.
#     Why: Seeing these values demonstrates how the reranker orders documents and what scores it assigns.
#     💡 Hint: Look at the reranked object structure. Check if it has .data or .matches attribute. Higher scores mean more relevant!



In [3]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 11.4 MB/s eta 0:00:00


In [34]:
import os
from google.colab import userdata

# Try to get PINECONE_KEY from environment variables first
PINECONE_API_KEY = os.getenv("PINECONE_KEY")

# If not found in environment, try Google Colab's userdata
if not PINECONE_API_KEY:
    PINECONE_API_KEY = userdata.get("PINECONE_KEY")

    # If still not found, prompt for authentication (Colab-specific)
    if not PINECONE_API_KEY:
        from pinecone_notebooks.colab import Authenticate
        Authenticate()
        # After authentication, the key should be available via userdata.get
        PINECONE_API_KEY = userdata.get("PINECONE_KEY")


In [36]:
from pinecone import Pinecone
# Use the PINECONE_API_KEY obtained from the previous cell
pc = Pinecone(api_key=PINECONE_API_KEY)

In [16]:
query = "Tell me about Apple's products"
documents = [
    "Apples are a healthy, crunchy fruit that usually ripens in the fall.", # Add a document about apple fruit
    "Apple Inc. has many popular products, including iPhone and iPad.", # Add a document about Apple company products
    "Apple cider is a refreshing drink that is also enjoyed hot.", # Add another fruit-related document
    "Apple Inc. has very popular computers such as MacBook.", # Add another company-related document
    "I like eating apples and drinking apple cider." # Add one more document (your choice)
]



In [29]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3 # e.g., 3
)

In [28]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    if matches is None:
        print("No reranked results found.")
        return
    for i, m in enumerate(matches):
        print(f'{i+1}, {m.score}, {m.document.text}')
        # Print the position (i+1), m.score, and m.document.text

show_reranked_results(query, reranked.data) # Fill in the correct attribute

Query: Tell me about Apple's products
1, 0.9129032, Apple Inc. has many popular products, including iPhone and iPad.
2, 0.41383246, Apple Inc. has very popular computers such as MacBook.
3, 0.017242778, Apples are a healthy, crunchy fruit that usually ripens in the fall.


In [37]:
# Part 2: Setup a Serverless Index for Medical Notes


# 1. Install data & model libraries

!pip install pandas torch transformers


#     What to do: Run this installation command in a notebook cell.
#     Why: You’ll use these libraries to load, embed, and manipulate medical note data.
#     💡 Hint: This might take a few minutes. Grab a coffee!


In [38]:
# 2. Import modules & define environment settings

import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings (these are defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws') # e.g., 'aws'
region = os.getenv('PINECONE_REGION', 'us-east-1') # e.g., 'us-east-1'

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index' # Give your index a name


    # What to do: Fill in the cloud provider (like ‘aws’), region (like ‘us-east-1’), and choose an index name.
    # Why: You’re configuring a serverless index tailored to your resource requirements and connecting the client in the proper cloud region.
    # 💡 Hint: Most Pinecone accounts use ‘aws’ and ‘us-east-1’. For index name, try something like medical-notes-index.



In [39]:
# 3. Create or recreate the index

# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384, # This matches our embedding model size
    metric='cosine', # Distance metric for similarity
    spec=spec
)


    # What to do: Fill in the dimension (384 for our model) and choose a metric (‘cosine’ is recommended).
    # Why: The index’s dimension must match the embedding vectors you’ll insert, otherwise upserts will fail.
    # 💡 Hint: The embedding model we’ll use outputs 384-dimensional vectors. Cosine similarity works well for text embeddings.


{
    "name": "medical-notes-index",
    "metric": "cosine",
    "host": "medical-notes-index-8crzmuh.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [44]:
import requests
import tempfile
import os
import pandas as pd

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/pinecone-io/examples/master/docs/data/sample_notes_data.jsonl" # Fixed GitHub raw URL
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

In [46]:

# 2. Preview the DataFrame

# # Show head of the DataFrame
print("Data shape:", df.shape) # Show number of rows and columns
df.head()


#     What to do: Fill in the correct pandas attribute to show DataFrame dimensions.
#     Why: Ensures you have the right columns (e.g., id , values , metadata ) before upserting.
#     💡 Hint: What pandas attribute shows the (rows, columns) of a DataFrame?




Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


In [47]:
# Part 4: Upsert Data into the Index


# 1. Instantiate index client & upsert

# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df) # Pass the DataFrame


#     What to do: Pass the DataFrame variable to the upsert function.
#     Why: This pushes all your note embeddings and metadata into Pinecone for later queries.
#     💡 Hint: What variable did you store the medical notes data in?


sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

{'upserted_count': 100}

In [48]:


# 2. Wait for availability

def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0 # What should this be?

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()


#     What to do: Fill in the condition - what number should the vector count be greater than?
#     Why: Ensures that upserted vectors are fully indexed before you attempt to query.
#     💡 Hint: We want to wait until there’s at least some vectors in the index. What’s the minimum?


Vector count:  100
Index ready!


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

In [49]:


# Part 5: Query & Embedding Function


# 1. Define your embedding function

def get_embedding(input_question):
  model_name = 'sentence-transformers/all-MiniLM-L6-v2'
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModel.from_pretrained(model_name)
  encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
  with torch.no_grad():
    model_output = model(**encoded_input)
    embedding = model_output.last_hidden_state[0].mean(dim=0) # Which dimension to average?
  return embedding


#     What to do: Fill in which dimension to average over (0 or 1).
#     Why: Converts incoming queries into the same vector space as your indexed notes.
#     💡 Hint: We want to average across the sequence length dimension to get a single vector per input.


In [52]:
# 2. Run a semantic search query

# Build a query to search
question = "patient has chest pain" # Ask a medical question
query = get_embedding(question).tolist()

# Get results
results = index.query(vector=[query], top_k=10, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)


#     What to do: Write a medical question and choose how many results to retrieve.
#     Why: Retrieves the most semantically similar notes from the index based on your clinical query.
#     💡 Hint: Try questions like “patient has chest pain” or “broken bone treatment”. Get 5-10 results.



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [54]:
# Part 6: Display & Rerank Clinical Notes


# 1. Display initial search results

def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}') # What field contains the score?
        print(f' Metadata: {match["metadata"]}') # What field contains metadata?
        print('')

show_results(question, sorted_matches)


#     What to do: Fill in the correct dictionary keys for score and metadata.
#     Why: Helps you see which notes were initially considered most relevant.
#     💡 Hint: Look at the structure of match objects from Pinecone query results.


Question: 'patient has chest pain'

Results:
   1. ID: P001
 Score: 0.734460175
 Metadata: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

   2. ID: P016
 Score: 0.48353824
 Metadata: {'condition': 'heart murmur', 'referral': 'cardiology'}

   3. ID: P0100
 Score: 0.446577758
 Metadata: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}

   4. ID: P095
 Score: 0.416666389
 Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

   5. ID: P047
 Score: 0.416666389
 Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

   6. ID: P003
 Score: 0.412105829
 Metadata: {'diagnosis': 'bronchitis', 'symptoms': 'cough, wheezing', 'treatment': 'antibiotics'}

   7. ID: P063
 Score: 0.385879159
 Metadata: {'diagnosis': 'pneumonia', 'symptoms': 'cough, fever', 'treatment': 'antibiotics'}

   8. ID: P090
 Score: 0.374950379
 Metadata: {'advice': 'stress management', 'symptoms': 'stress, burnout'}

   9. ID: P042
 Score: 0.374950379

In [55]:
# 2. Prepare documents for reranking

# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]


#     What to do: Fill in the correct key to access the metadata from each match.
#     Why: Constructs a field summarizing each note’s metadata for the reranker to use when rescoring.
#     💡 Hint: What field did you just use in the previous step to print metadata?


In [56]:
# 3. Execute serverless reranking

# Define a more specific query for reranking
refined_query = "diabetes treatment plan" # Make a more specific medical question

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3, # How many top results do you want?
    return_documents=True,
)


#     What to do: Create a more specific medical query and choose how many reranked results to return.
#     Why: Reranking uses the refined query and metadata field to reorder notes by their new relevance scores.
#     💡 Hint: Try “patient needs knee surgery” or “diabetes treatment plan”. Get 2-3 top results.


In [58]:
# 4. Show reranked results

def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match['score']}') # What attribute contains the reranking score?
        print(f' Reranking Field: {match.document['reranking_field']}') # What contains the searchable field?
        print('')
show_reranked_results(refined_query, reranked_results.data) # What attribute contains the results?


#     What to do: Fill in the attributes for the reranking score, searchable field, and results collection.
#     Why: Allows you to compare how the reranker improves result ordering against the original search.
#     💡 Hint: Check the reranked object structure - look for .score, field names, and .data or similar.


Question: 'diabetes treatment plan'

Reranked Results:
   1. ID: P095
 Score: 0.005260852
 Reranking Field: symptoms: back pain; treatment: physical therapy

   2. ID: P047
 Score: 0.005260852
 Reranking Field: symptoms: back pain; treatment: physical therapy

   3. ID: P003
 Score: 0.0006070756
 Reranking Field: diagnosis: bronchitis; symptoms: cough, wheezing; treatment: antibiotics



In [59]:
# 5. Clean up (optional)

# Delete the index to save resources
pc.delete_index(name=index_name)


#     What to do: Run this when you’re done to avoid unnecessary charges.
#     Why: Serverless indexes cost money when they contain data, so clean up after experiments.
#     💡 Hint: You can always recreate the index later if needed!


In [ ]:


# 🎯Success Criteria

# To complete this challenge successfully, you should:

# ✅ Successfully authenticate with Pinecone
# ✅ Run basic document reranking and see sensible results
# ✅ Create and populate a serverless index with medical notes
# ✅ Execute semantic search queries on medical data
# ✅ Compare original search results with reranked results
# ✅ Understand how reranking improves search relevance